# MLflow: from trials to a registered model
**Internet of Production – AI, week 3, block 5** · about 25 min · work in pairs

**Thread:** The transfer test showed that the frozen rule works on all six printers of 2026. But in the test you tried several limits and models and kept the numbers in your head or on paper. Next week someone asks: *"Which model runs in the monitor, how was it tested, and can we get back to the old one?"* For that you need a record.

**Goal of this notebook**
1. log every trial as a **run** (parameters, metrics, the model itself) in MLflow,
2. compare the runs in the **web interface**,
3. **register** the chosen model with a version and the alias `production`, so that the service of the next notebook can load it by name.

**Theory in one sentence each:** *Tracking* records what was tried. A *model registry* gives a model a name and numbered versions. An *alias* is a movable label (here `production`) that decides which version is live, so that a roll-back is a change of the label and not of code.

In [ ]:
import sys, subprocess, time, logging, warnings
sys.path.insert(0, '..')                                   # blockage_core.py lies in the folder above
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import mlflow, mlflow.pyfunc
logging.getLogger('mlflow').setLevel(logging.ERROR); warnings.filterwarnings('ignore')     # keep the output readable
from blockage_core import *

DATA_OLD = Path('../../00_Data/printfarm')
DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"      # one database for the whole class folder (and for the service)
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT = 'printfarm-transfer'
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(STORE / 'mlartifacts').as_uri())
mlflow.set_experiment(EXPERIMENT)

refs = fit_references(DATA_OLD)                             # frozen references of weeks 1 to 3 (printer 1, 2024/25, first 70 %)
printers = {k: make_blockage_data(DATA_NEW, k) for k in range(6)}   # the six printers of 2026
print(f"reference: E120 = {refs['mu']:.3f} +/- {refs['sd']:.3f} Wh/min, z limit {refs['z_limit']:.2f}; {len(printers)} printers loaded; tracking in {TRACKING_URI}")

## Step 1: Log runs
`log_run` evaluates a model on all six printers of 2026 (the model stays frozen) and writes three things to MLflow: the **parameters** that define the trial, the **metrics** and the **model** as an artifact.

Metrics: `auc` (mean over the printers), `announced_share` (blockages with at least one alarm in the 6 hours before), `false_episodes_per_week` (mean per printer). Three metrics are deliberately kept apart: a single number would hide the trade-off between missed blockages and false alarms.

Run the z-value rule with three limits and the KNN with two decision thresholds. Choose a **lower** and a **higher** z limit around the default.

In [ ]:
def log_run(name, model, params):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        metrics = evaluate_on_farm(printers, model)
        mlflow.log_metrics(metrics)
        mlflow.pyfunc.log_model(name='model', python_model=model, input_example=refs['ref'][FEATURES].head(2).astype(float),
                                pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
    return metrics

for z_limit in [___, round(refs['z_limit'], 2), ___]:          # a lower and a higher limit
    model = ZRuleModel(refs['mu'], refs['sd'], z_limit)
    log_run(f'zrule z>{z_limit}', model, {'model': 'zrule', 'z_limit': z_limit, 'mu': round(refs['mu'], 4), 'sd': round(refs['sd'], 4)})

for threshold in [0.5, ___]:                                    # the default and a lower threshold
    model = KnnModel(refs['knn'], threshold)
    log_run(f'knn k=25 thr={threshold}', model, {'model': 'knn', 'k': 25, 'threshold': threshold})
print('5 runs logged')

## Step 2: Compare the runs
Read the runs back from the database and plot the trade-off. Every point is one trial; the best trial is **up and to the left** (many blockages announced, few false alarms).

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=['start_time ASC'])
table = runs[['tags.mlflow.runName', 'metrics.auc', 'metrics.announced_share', 'metrics.false_episodes_per_week']].copy()
table.columns = ['run', 'AUC', 'announced share', 'false episodes / week']
display(table.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(table['false episodes / week'], table['announced share'])
for _, r in table.iterrows():
    ax.annotate(r['run'], (r['false episodes / week'], r['announced share']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('false alarm episodes per printer and week'); ax.set_ylabel('share of blockages announced (6 h)'); ax.grid(alpha=.3)
plt.show()

**Which run would you put into production? What does the answer depend on? Why is AUC alone not enough to choose?**

*Your answer:*

## Step 3: The web interface
Start the MLflow UI and open it. **Locally / on the pool PC:** `http://localhost:5000`. **On Binder:** click the link that appears below the cell (it points to the same server behind the Binder proxy, `…/proxy/5000/`). Starting takes a few seconds.

Tasks in the browser (5 min):
1. Open the experiment `printfarm-transfer`, select two runs and click **Compare**.
2. Open one run: find parameters, metrics and the model artifact (look at `MLmodel` and `requirements.txt`).
3. Sort the table by `false_episodes_per_week`.

In [ ]:
import os, requests
from IPython.display import HTML, display
env = dict(os.environ, MLFLOW_SERVER_ENABLE_JOB_EXECUTION='false')    # no background job workers: saves about 2 GB of memory (decisive on Binder, which has 2 GB)
ui = subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui', '--backend-store-uri', TRACKING_URI, '--host', '127.0.0.1', '--port', '5000', '--workers', '1',
                       '--allowed-hosts', '*', '--cors-allowed-origins', '*'],     # Binder forwards its own host name; without this MLflow answers 'Invalid Host header'
                      env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(90):                                                     # wait until the server answers
    try:
        requests.get('http://127.0.0.1:5000/', timeout=1); break
    except requests.RequestException:
        time.sleep(1)
prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')                # set on Binder / JupyterHub, e.g. /user/<name>/
print('MLflow UI is running. Local: http://localhost:5000')
if prefix:
    display(HTML(f'<a href="{prefix}proxy/5000/" target="_blank">Binder / JupyterHub: open the MLflow UI</a>'))

**What do you find in the run that you did not log yourself? Which information would you need to reproduce the result in a year?**

*Your answer:*

## Step 4: Register the models
Registering gives the model a **name** and a **version number**. The alias `production` marks the version that the service loads. Register the z-value rule with the limit of your choice and the KNN with threshold 0.5.

In [ ]:
CHOSEN_Z_LIMIT = ___                                           # your decision from step 2

client = mlflow.MlflowClient()
def register(name, model, params):
    with mlflow.start_run(run_name=f'register {name}'):
        mlflow.log_params(params)
        info = mlflow.pyfunc.log_model(name='model', python_model=model, input_example=refs['ref'][FEATURES].head(2).astype(float),
                                       pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
    version = mlflow.register_model(info.model_uri, name)
    client.set_registered_model_alias(name, '___', version.version)    # which alias does the service look for?
    print(f'{name}: version {version.version} has the alias production')

register('blockage-zrule', ZRuleModel(refs['mu'], refs['sd'], CHOSEN_Z_LIMIT), {'model': 'zrule', 'z_limit': CHOSEN_Z_LIMIT})
register('blockage-knn', KnnModel(refs['knn'], 0.5), {'model': 'knn', 'k': 25, 'threshold': 0.5})

Check in the browser under **Models**: both models should appear with their version and the alias. Then stop the UI (the registry stays in `mlflow.db`).

In [ ]:
ui.terminate()
for m in client.search_registered_models():
    print(m.name, {a: v for a, v in m.aliases.items()})

**The registry stores the model, but not the *features*. What can go wrong when training code and serving code compute E120 differently? Where is that prevented in this course?**

*Your answer:*